# Dask: Parallel Computing for Python at Scale

## What Is Dask?

Imagine you have a 100GB CSV file. `pd.read_csv('file.csv')` would crash your laptop — not enough RAM.  
**Dask** solves this by reading the file in chunks, processing each chunk, and combining results — like reading a book 100 pages at a time instead of all at once.

**Dask** is a parallel computing library that:
- Extends **pandas** (`dask.dataframe`) and **NumPy** (`dask.array`) to datasets larger than RAM
- Runs computations in **parallel** on multiple CPU cores
- Can scale to **distributed clusters** with `dask.distributed`
- Uses **lazy evaluation**: builds a computation graph first, executes when you call `.compute()`

## Resources

- **Docs**: [https://docs.dask.org/](https://docs.dask.org/)
- **GitHub**: [https://github.com/dask/dask](https://github.com/dask/dask)
- **YouTube — Dask tutorial**: [https://www.youtube.com/watch?v=RA_2qdipVng](https://www.youtube.com/watch?v=RA_2qdipVng)

## Installation

```bash
pip install dask[complete]   # all features (dataframe, array, distributed)
pip install dask             # core only
pip install dask[dataframe]  # just dask.dataframe
```

In [ ]:
import numpy as np
import pandas as pd
import time, os, tempfile

try:
    import dask
    import dask.dataframe as dd
    import dask.array as da
    from dask import delayed
    DASK_AVAILABLE = True
    print(f"Dask version: {dask.__version__}")
except ImportError:
    DASK_AVAILABLE = False
    print("Dask not installed — simulated output. Install: pip install dask[complete]")

# Generate a realistic synthetic dataset
np.random.seed(42)
N = 500_000  # 500K rows to show performance difference
df_pandas = pd.DataFrame({
    'customer_id': np.arange(N),
    'age': np.random.randint(18, 80, N),
    'income': np.random.exponential(50000, N).astype(int),
    'region': np.random.choice(['North', 'South', 'East', 'West'], N),
    'product': np.random.choice(['A', 'B', 'C', 'D'], N),
    'spend': np.random.exponential(200, N).round(2),
    'is_churned': np.random.randint(0, 2, N),
})
print(f"Dataset: {N:,} rows × {len(df_pandas.columns)} columns")
print(f"Memory (pandas): {df_pandas.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## Core Concept 1: Lazy Evaluation and Partitions

**Lazy evaluation**: Dask builds a computation plan (task graph) but doesn't execute until you call `.compute()`.  
**Partitions**: Dask splits data into chunks. Each chunk is a pandas DataFrame processed independently.

In [ ]:
if DASK_AVAILABLE:
    # Convert pandas → dask (split into 4 partitions)
    ddf = dd.from_pandas(df_pandas, npartitions=4)

    print(f"Dask DataFrame:")
    print(f"  Partitions: {ddf.npartitions}")
    print(f"  Rows per partition: ~{N//4:,}")
    print(f"  Columns: {list(ddf.columns)}")
    print()

    # This is LAZY — no computation yet, just builds a plan
    lazy_result = ddf.groupby('region')['spend'].mean()
    print("Lazy result (not computed yet):")
    print(type(lazy_result), "\n")

    # .compute() triggers actual execution
    t0 = time.time()
    result = lazy_result.compute()  # NOW it runs!
    dask_time = time.time() - t0

    # Compare with pandas
    t0 = time.time()
    result_pandas = df_pandas.groupby('region')['spend'].mean()
    pandas_time = time.time() - t0

    print("Spend by region:")
    print(result.round(2).to_frame())
    print(f"\nDask time:   {dask_time*1000:.1f} ms")
    print(f"Pandas time: {pandas_time*1000:.1f} ms")
    print("(Dask overhead is noticeable on small data — shines on 10GB+)")

else:
    print("Dask DataFrame concepts (simulated):")
    print()
    print("  ddf = dd.from_pandas(df, npartitions=4)")
    print("  # Creates: 4 partition DataFrames, each 125K rows")
    print()
    print("  # LAZY — builds computation graph, no data moved yet")
    print("  lazy = ddf.groupby('region')['spend'].mean()")
    print("  print(type(lazy))  # <class 'dask.dataframe.core.Series'>")
    print()
    print("  # .compute() executes the graph in parallel")
    print("  result = lazy.compute()")
    print("  # North: 198.7  South: 201.3  East: 199.8  West: 200.5")

## Core Concept 2: dask.array — Parallel NumPy

In [ ]:
if DASK_AVAILABLE:
    # Create a large dask array from numpy
    x_np = np.random.random((10000, 1000))  # 10K × 1K = 10M elements
    x_da = da.from_array(x_np, chunks=(1000, 1000))  # each chunk: 1K rows

    print(f"Dask array: {x_da.shape}, chunks={x_da.chunks[0][0]} rows")

    # Operations look exactly like NumPy
    mean_val    = x_da.mean().compute()
    std_val     = x_da.std().compute()
    row_sums    = x_da.sum(axis=1).compute()  # sum across columns for each row
    normalized  = ((x_da - x_da.mean()) / x_da.std()).compute()

    print(f"Mean: {mean_val:.4f}  Std: {std_val:.4f}")
    print(f"Row sums (first 5): {row_sums[:5].round(2)}")
    print(f"Normalized shape: {normalized.shape}")
    print()

    # The task graph — visualize the computation plan
    result = (x_da * 2 + 1).mean()
    print(f"Task graph for (x*2+1).mean():")
    print(f"  Number of tasks: {len(result.__dask_graph__())}")
    print(f"  Tasks: multiply → add → partial means → final mean")
    print("  (Run result.visualize() to see the actual graph if graphviz installed)")

else:
    print("dask.array example (simulated):")
    print()
    print("  import dask.array as da")
    print("  x = da.from_array(np.random.random((10000, 1000)), chunks=(1000, 1000))")
    print("  # Looks just like numpy:")
    print("  mean  = x.mean().compute()     # → 0.5000")
    print("  norms = (x * 2 + 1).mean().compute()  # → 2.0000")
    print("  # But runs in parallel across 10 chunks!")

## Core Concept 3: Reading Large Files with Dask

In [ ]:
if DASK_AVAILABLE:
    # Write multiple CSV files (simulating a real data lake scenario)
    tmp_dir = tempfile.mkdtemp()
    chunk_size = 50_000

    for i, start in enumerate(range(0, N, chunk_size)):
        chunk = df_pandas.iloc[start:start+chunk_size]
        chunk.to_csv(os.path.join(tmp_dir, f'data_{i:04d}.csv'), index=False)

    n_files = len(os.listdir(tmp_dir))
    print(f"Created {n_files} CSV files in {tmp_dir}")

    # Read ALL CSVs at once with dask (lazy!)
    ddf_from_csv = dd.read_csv(os.path.join(tmp_dir, 'data_*.csv'))
    print(f"Dask DataFrame from {n_files} CSVs: {ddf_from_csv.npartitions} partitions")

    # Complex aggregation across all files
    result = (
        ddf_from_csv
        .assign(spend_per_income=lambda df: df['spend'] / df['income'].clip(lower=1))
        .groupby(['region', 'product'])
        .agg({'spend': ['mean', 'sum'], 'is_churned': 'mean'})
        .compute()
    )
    result.columns = ['avg_spend', 'total_spend', 'churn_rate']
    print("\nAggregation result (region × product):")
    print(result.round(2).head(8))

else:
    print("Reading multiple CSVs with Dask (simulated):")
    print()
    print("  # Read all CSV files matching the pattern (lazy!)")
    print("  ddf = dd.read_csv('data/logs_*.csv')")
    print("  # Dask figures out: 500 files × 100K rows = 50M rows total")
    print()
    print("  # Same pandas API — groupby, merge, filter")
    print("  result = (")
    print("      ddf")
    print("      .groupby('region')")
    print("      .agg({'spend': 'sum', 'is_churned': 'mean'})")
    print("      .compute()  # ← executes across all 500 files in parallel")
    print("  )")

## Core Concept 4: dask.delayed — Parallelize Any Python Function

In [ ]:
if DASK_AVAILABLE:
    from dask import delayed, compute

    # @delayed turns a regular function into a lazy task
    @delayed
    def process_chunk(chunk_df, feature_name):
        """Process a chunk of data — runs in parallel."""
        stats = {
            'feature': feature_name,
            'mean': chunk_df[feature_name].mean(),
            'std':  chunk_df[feature_name].std(),
            'p95':  chunk_df[feature_name].quantile(0.95),
        }
        return stats

    # Build a list of delayed tasks
    features = ['age', 'income', 'spend']
    tasks = []
    for feature in features:
        task = process_chunk(df_pandas, feature)  # LAZY — not computed yet
        tasks.append(task)

    # Execute all tasks in parallel
    t0 = time.time()
    results = compute(*tasks)  # runs all 3 concurrently
    elapsed = time.time() - t0

    print(f"Parallel feature stats ({elapsed*1000:.0f}ms):")
    for stat in results:
        print(f"  {stat['feature']:10s}: mean={stat['mean']:.1f}  std={stat['std']:.1f}  p95={stat['p95']:.1f}")

else:
    print("dask.delayed — parallelize any function (simulated):")
    print()
    print("  @delayed")
    print("  def process_chunk(df, feature):")
    print("      return {'mean': df[feature].mean(), 'std': df[feature].std()}")
    print()
    print("  # Build tasks (lazy)")
    print("  tasks = [process_chunk(chunk, 'spend') for chunk in chunks]")
    print()
    print("  # Execute all in parallel")
    print("  results = dask.compute(*tasks)")
    print("  # → all chunks processed simultaneously")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Calling `.compute()` too early | Loses parallelism | Chain operations before computing |
| Too many small partitions | Scheduler overhead > compute | Use larger partitions (100MB each is good) |
| Too few partitions | Single-core bottleneck | `npartitions = num_cores × 2-4` |
| Using `apply(lambda)` | Slow (row by row) | Use vectorized operations like pandas |
| `len(ddf)` forces compute | Slow | Use `ddf.shape[0].compute()` or avoid if possible |
| Mixing pandas and dask | Type errors | Convert: `ddf = dd.from_pandas(df, npartitions=4)` |
| Not enough memory for collect | OOM on `.compute()` | Filter/aggregate more before computing |

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "Dask vs Apache Spark — which would you choose for a 50GB dataset?",
     "a": """It depends on your infrastructure and team:

Choose Dask when:
- Your team knows Python/pandas and wants minimal new learning
- You work on a single machine with many cores (32+ CPU), using out-of-core
- Your data fits on one machine's disk (not a cluster)
- You need tight NumPy/scikit-learn integration
- You want to prototype quickly

Choose Spark when:
- Data is truly distributed across a cluster (multi-terabyte)
- You need mature SQL engine (Spark SQL is excellent)
- Your org already has a Spark/Hadoop cluster
- You need streaming (Spark Structured Streaming)
- Your data engineers are already Spark experts

For 50GB on a workstation with 256GB RAM + many cores:
→ Dask is simpler, pandas API, no cluster setup

For 50GB across 10 worker machines:
→ Either works; Spark more battle-tested for distributed SQL"""},

    {"q": "What does 'lazy evaluation' mean in Dask and why is it beneficial?",
     "a": """Lazy evaluation means: operations build a computation graph (recipe) but don't run until .compute() is called.

Benefits:
1. Optimization: Dask can optimize the graph before executing
   (e.g., push filters before expensive joins, fuse redundant operations)

2. Memory efficiency: only data needed for the final result is loaded
   ddf.read_csv().filter(col > 0).mean()  # only reads rows where col > 0

3. Flexibility: you can build complex computation plans and then
   choose when/how to execute them

4. Parallelism planning: Dask sees the full graph and can schedule
   independent tasks to run simultaneously

vs Eager execution (pandas):
  df.filter() → creates new DataFrame immediately (memory used)
  df.groupby() → another DataFrame
  Each step allocates memory and runs sequentially.

Dask builds the full recipe, then executes optimally."""},

    {"q": "When is Dask SLOWER than pandas and why?",
     "a": """Dask is slower than pandas when:

1. Small data (fits easily in RAM):
   Dask has task scheduling overhead (~milliseconds per task)
   pandas executes directly in C with no overhead
   Rule of thumb: use pandas if data < 10GB on a machine with 64GB RAM

2. Many small partitions:
   10,000 partitions of 1MB each → 10,000 tasks → scheduler bottleneck
   Fix: repartition to fewer, larger chunks

3. Operations that require global sort (e.g., sorting all data):
   Dask must shuffle data between partitions → expensive
   These are called 'wide dependencies' or 'shuffles'

4. Iterative algorithms (ML training loops):
   Dask reloads data each iteration; pandas can cache it
   Use .persist() to keep computed results in memory

Summary: Dask excels at data larger than RAM. Use pandas for smaller data."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Feature | Dask API |
|---------|----------|
| From pandas | `dd.from_pandas(df, npartitions=4)` |
| Read CSVs | `dd.read_csv('path/*.csv')` |
| Read parquet | `dd.read_parquet('path/*.parquet')` |
| Filter | `ddf[ddf['col'] > 0]` |
| GroupBy | `ddf.groupby('col').agg({'x': 'mean'})` |
| Execute | `.compute()` (returns pandas) |
| Keep in memory | `.persist()` |
| NumPy-like | `da.from_array(x, chunks=(1000,))` |
| Parallelize functions | `@delayed` + `dask.compute(*tasks)` |

### Next Steps
1. **Dask tutorial**: [https://tutorial.dask.org/](https://tutorial.dask.org/)
2. **Dask distributed**: [https://distributed.dask.org/](https://distributed.dask.org/)
3. **Next**: Learn Polars — a faster alternative for in-memory DataFrame operations